# Step 04 - Research + Writer

Goal: pass research output to a writer agent.

## What's New in This Step

- Step 03 had one research agent with a tool.
- This step adds a second agent (`writer_agent`) and task chaining via `context=[research_task]`.
- You now see a pipeline pattern: gather evidence first, then generate content from that evidence.

### Setup

In [ ]:
import os
from dotenv import load_dotenv
from crewai import LLM, Agent, Crew, Task
from crewai_tools import SerperDevTool

load_dotenv()

TOPIC = "Platform Engineering Best Practices"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")

if not openrouter_api_key:
    raise ValueError("Missing OPENROUTER_API_KEY")

if not serper_api_key:
    raise ValueError("Missing SERPER_API_KEY")

### Create LLM and Agents

In [ ]:
# LLM: shared across both agents so behavior differences come from prompts/roles.
llm = LLM(
    model="openai/gpt-4o",
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

# Built-in tool used by the research agent for fresh sources.
search_tool = SerperDevTool()

# Agent 1: collects factual context with citations.
research_agent = Agent(
    role="Research Analyst",
    goal="Find latest updates on {topic}",
    backstory="You give short, factual, source-backed summaries.",
    llm=llm,
    tools=[search_tool],
    verbose=False,
)

# Agent 2: turns research notes into a learner-friendly article draft.
writer_agent = Agent(
    role="Content Writer",
    goal="Write a simple blog post on {topic}",
    backstory="You write beginner-friendly technical content.",
    llm=llm,
    verbose=False,
)

### Define Tasks with Context

In [ ]:
# Task 1: gather recent source-backed notes.
research_task = Task(
    description="Research {topic} from 2026 onward with source links.",
    expected_output="A short research summary with sources.",
    agent=research_agent,
)

# Task 2: generate Markdown using the output of task 1 as context.
writing_task = Task(
    description="Write a clear blog post from the research notes.",
    expected_output="A markdown blog post.",
    agent=writer_agent,
    context=[research_task],
)

### Create Crew with Both Agents and Tasks

In [ ]:
crew = Crew(
    agents=[research_agent, writer_agent],
    tasks=[research_task, writing_task],
    verbose=False,
)

### Kickoff the Crew with the topic input

In [ ]:
result = crew.kickoff(inputs={"topic": TOPIC})
print(getattr(result, "raw", str(result)))

### Recap
- LLM did: reason over research notes to draft the article.
- Agents did: split responsibilities between research and writing.
- Task enforced: execution order and Markdown output expectation.